# 07 – Phase 1 results: what we learned about NeuReduce

The presentation notebook. Only results that told us something about the model, one figure per finding, consistent
style. Every figure title states the finding; every figure is saved to `figures/final/` for the slides.

**Inputs** (all produced by the earlier notebooks):
`data/predictions_test.csv`, `data/predictions_train.csv` (common setup) · `taxonomy.csv` (`04`) ·
`probe_results_full.csv` or `probe_results.csv` (`05`). Section 5 computes one UMAP projection (~1 min).

| | Finding | Figure |
|---|---|---|
| 1 | The model is accurate overall, but its errors are not random | `01_overall` |
| 2 | Rare kinds of input fail far more often | `02_rare_inputs` |
| 3 | Answers that barely simplify, and inputs with few terms, are hard | `03_difficulty` |
| 4 | Memorization: unseen answers fail more – but cause few of the errors | `04_memorization` |
| 5 | Most mistakes are near-misses | `05_mistakes` |
| 6 | Two-variable AND/OR answers get collapsed to one variable | `06_bitwise` |
| 7 | Confidence flags about half the errors – the rest are confident | `07_confidence` |
| 8 | Meaning-preserving rewrites break most correct answers | `08_rewrites` |
| 9 | …in every kind of input | `09_fragile_everywhere` |
| 10 | With unfamiliar variable names, the model invents variables | `10_invented_variables` |
| 11 | The training data has a fixed operand order | `11_convention` |
| 12 | On a map of all inputs, only the 3-variable inputs stand out | `12_map` |

In [ ]:
import ast
import itertools
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "predictions_test.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
import neureduce as nr

HERE = Path.cwd()
OUT = HERE / "figures" / "final"
OUT.mkdir(parents=True, exist_ok=True)

# ---- data
test = pd.read_csv(ROOT / "data" / "predictions_test.csv")
train = pd.read_csv(ROOT / "data" / "predictions_train.csv")
for d in (test, train):
    d["error"] = ~d["correct"]
test["var_pair"] = ["".join(sorted(set(re.findall(r"[a-z]", s)))) for s in test.src]
test["input_type"] = np.select([test.src_n_vars == 3, test.var_pair != "xy"],
                               ["three variables", "other variable pair"], "variables x, y")
train_answers = Counter(train.trg)
test["answer_seen"] = test.trg.map(train_answers).fillna(0).astype(int)
BASE = test.error.mean()

tax = pd.read_csv(HERE / "taxonomy.csv")
probe_file = HERE / "probe_results_full.csv"
if not probe_file.exists():
    probe_file = HERE / "probe_results.csv"
probes = pd.read_csv(probe_file)
probes["ok"] = probes.new_verdict.isin(["exact", "equivalent"])
print(f"test {len(test):,} · train {len(train):,} · mistakes {len(tax):,} · rewrites from {probe_file.name} ({len(probes):,})")

# ---- style
C = {"correct": "#2a9d8f", "wrong": "#e63946", "neutral": "#c9ced6", "accent": "#457b9d",
     "dark": "#1d3557", "soft": "#a8dadc", "warm": "#f4a261"}
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 220, "savefig.bbox": "tight", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#888888",
    "axes.grid": True, "grid.color": "#e6e6e6", "axes.axisbelow": True,
    "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "xtick.color": "#444444", "ytick.color": "#444444",
})
GOOD_BAD = LinearSegmentedColormap.from_list("good_bad", [C["wrong"], "#f6e3b4", C["correct"]])


def headline(fig, title, subtitle=None):
    """Finding as figure title (left-aligned) plus a one-line explanation; drawn when the figure is saved."""
    fig._headline = (title, subtitle)


def save(fig, name):
    h = fig.get_figheight()
    title, subtitle = getattr(fig, "_headline", (None, None))
    fig.tight_layout(rect=(0, 0, 1, 1 - (0.8 if subtitle else 0.5) / h))
    if title:
        fig.text(0.01, 1 - 0.22 / h, title, ha="left", va="center", fontsize=13, fontweight="bold", color=C["dark"])
    if subtitle:
        fig.text(0.01, 1 - 0.52 / h, subtitle, ha="left", va="center", fontsize=9.5, color="#555555")
    fig.savefig(OUT / f"{name}.png")
    plt.show()


def wilson(k, n, z=1.96):
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = k / n
    d = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return centre - half, centre + half


def rates(data, by, col="error"):
    t = data.groupby(by, observed=True)[col].agg(n="size", k="sum")
    t["rate"] = t.k / t.n
    t["lo"], t["hi"] = wilson(t.k, t.n)
    return t


def pct(x, digits=0):
    return f"{100 * x:.{digits}f}%"

## 1 · Accurate overall – but not uniformly

*What you see:* how every output was judged, for examples the model trained on and for unseen test examples.
"Equivalent" = correct, but written differently from the dataset's answer (checked mathematically).

In [ ]:
order = [("exact", "correct (same answer)", C["correct"]), ("equivalent", "correct (rewritten)", C["soft"]),
         ("wrong", "wrong", C["wrong"]), ("invalid", "not an expression", C["dark"])]
fig, ax = plt.subplots(figsize=(9, 2.6))
for y, (name, d) in enumerate([("training examples", train), ("unseen test examples", test)]):
    left = 0
    for verdict, label, color in order:
        share = (d.verdict == verdict).mean()
        ax.barh(y, share, left=left, color=color, height=0.55, label=label if y == 0 else None)
        if share > 0.04:
            ax.text(left + share / 2, y, pct(share, 1), ha="center", va="center", color="white", fontweight="bold")
        left += share
    ax.text(1.01, y, f"n = {len(d):,}", va="center", fontsize=9, color="#555555")
ax.set_yticks([0, 1], ["training examples", "unseen test examples"])
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(lambda v, _: pct(v))
ax.grid(False)
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.18), fontsize=9)
headline(fig, f"Correct on {pct(1 - train.error.mean())} of training and {pct(1 - test.error.mean())} of unseen examples",
         "Errors are about three times as frequent on unseen inputs – the rest of this notebook shows where and why.")
save(fig, "01_overall")

## 2 · Rare kinds of input fail far more often

*What you see:* error rate on unseen examples by input type, with the share of the **training data** of each type.
The black whiskers are 95% confidence intervals.

In [ ]:
train["input_type"] = np.select([train.src_n_vars == 3,
                                 train.src.map(lambda s: set(re.findall(r"[a-z]", s)) != {"x", "y"})],
                                ["three variables", "other variable pair"], "variables x, y")
types = ["variables x, y", "other variable pair", "three variables"]
t = rates(test, "input_type").reindex(types)
train_share = train.input_type.value_counts(normalize=True).reindex(types)

fig, ax = plt.subplots(figsize=(8, 3.6))
colors = [C["accent"], C["warm"], C["wrong"]]
ax.bar(types, t.rate, color=colors, width=0.6)
ax.errorbar(types, t.rate, yerr=[t.rate - t.lo, t.hi - t.rate], fmt="none", ecolor="black", capsize=4, lw=1)
for i, typ in enumerate(types):
    ax.text(i, t.hi.iloc[i] + 0.02, f"{pct(t.rate.iloc[i])} wrong", ha="center", fontweight="bold", color=colors[i])
    ax.text(i, -0.075, f"{pct(train_share.iloc[i], 1)} of training · n = {t.n.iloc[i]:,} test",
            ha="center", fontsize=8.5, color="#555555", transform=ax.get_xaxis_transform())
ax.axhline(BASE, color="#888888", ls="--", lw=1)
ax.text(2.45, BASE, f"overall {pct(BASE, 1)}", va="bottom", ha="right", fontsize=8.5, color="#777777")
ax.set_ylim(0, 0.62)
ax.yaxis.set_major_formatter(lambda v, _: pct(v))
ax.set_ylabel("share of wrong outputs")
ax.tick_params(axis="x", pad=18)

same_terms = test[test.src_n_terms.isin([6, 7])].groupby(["src_n_terms", "input_type"]).error.mean().unstack()
check = ", ".join(f"{k} terms: {pct(r['three variables'])} vs {pct(r['variables x, y'])}" for k, r in same_terms.iterrows())
headline(fig, "Inputs with three variables fail five times as often",
         f"Only 1% of training data has three variables. Same length, different variables – {check}.")
save(fig, "02_rare_inputs")

## 3 · What makes an input hard: few terms, and answers that stay long

*What you see:* error rate (line, with 95% band) against the number of terms in the input (left) and the length of the
correct answer (right). Grey bars: how many test examples each point is based on. Points with fewer than 30 examples
are left out.

In [ ]:
def rate_line(ax, data, col, bins=None, labels=None, xlabel="", min_n=30):
    key = pd.cut(data[col], bins, labels=labels, right=False) if bins is not None else data[col]
    t = rates(data.assign(key=key), "key")
    t = t[t.n >= min_n]
    x = np.arange(len(t))
    ax2 = ax.twinx()
    ax2.bar(x, t.n, color=C["neutral"], alpha=0.45, width=0.7)
    ax2.set_yticks([]); ax2.grid(False); ax2.spines[["top", "right", "left"]].set_visible(False)
    ax.set_zorder(ax2.get_zorder() + 1); ax.patch.set_visible(False)
    ax.fill_between(x, t.lo, t.hi, color=C["wrong"], alpha=0.15)
    ax.plot(x, t.rate, "o-", color=C["wrong"], lw=2)
    for xi, r in zip(x, t.rate):
        ax.text(xi, r + 0.025, pct(r), ha="center", fontsize=8.5, color=C["wrong"], fontweight="bold")
    ax.set_xticks(x, [str(i) for i in t.index])
    ax.set_xlabel(xlabel)
    ax.yaxis.set_major_formatter(lambda v, _: pct(v))
    ax.set_ylim(0, max(0.4, t.hi.max() + 0.06))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
rate_line(axes[0], test, "src_n_terms", xlabel="terms in the input")
axes[0].set_ylabel("share of wrong outputs")
axes[0].set_title("Fewer terms in the input → more errors")
rate_line(axes[1], test, "trg_len", bins=[0, 3, 6, 9, 12, 15, 18, 21],
          labels=["0-2", "3-5", "6-8", "9-11", "12-14", "15-17", "18-20"], xlabel="length of the correct answer (characters)")
axes[1].set_title("Long answers (little simplification) → more errors")
headline(fig, "Hard inputs: few terms, or an answer that barely simplifies",
         "Input length itself matters little once the number of terms is known; nesting depth has no effect (always 1-2).")
save(fig, "03_difficulty")

## 4 · Memorization is real – but explains only a small part of the errors

*What you see (left):* error rate by how often the exact correct answer appeared in the training data.
*(Right):* all wrong outputs, split by whether their correct answer was ever seen in training.

In [ ]:
edges, labels = [0, 1, 2, 5, 20, 100, np.inf], ["never", "1", "2-4", "5-19", "20-99", "100+"]
t = rates(test.assign(freq=pd.cut(test.answer_seen, edges, right=False, labels=labels)), "freq")

fig, (ax, axr) = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [2.2, 1]})
x = np.arange(len(t))
ax.bar(x, t.rate, color=[C["wrong"]] + [C["accent"]] * (len(t) - 1), width=0.65)
ax.errorbar(x, t.rate, yerr=[t.rate - t.lo, t.hi - t.rate], fmt="none", ecolor="black", capsize=3, lw=1)
for xi, (r, n, hi) in enumerate(zip(t.rate, t.n, t.hi)):
    ax.text(xi, hi + 0.01, f"{pct(r)}\nn={n:,}", ha="center", fontsize=8.5)
ax.set_xticks(x, labels)
ax.set_xlabel("times the correct answer appeared in training")
ax.set_ylabel("share of wrong outputs")
ax.yaxis.set_major_formatter(lambda v, _: pct(v))
ax.set_ylim(0, t.hi.max() + 0.08)
ax.set_title("The more often an answer was seen, the fewer errors")

errs = test[test.error]
unseen_share = (errs.answer_seen == 0).mean()
in_train = errs.pred.isin(train_answers).mean()
axr.barh([0], [unseen_share], color=C["wrong"], height=0.5)
axr.barh([0], [1 - unseen_share], left=[unseen_share], color=C["accent"], height=0.5)
axr.text(unseen_share / 2, 0, pct(unseen_share), ha="center", va="center", color="white", fontweight="bold")
axr.text(unseen_share + (1 - unseen_share) / 2, 0, pct(1 - unseen_share), ha="center", va="center", color="white", fontweight="bold")
axr.text(unseen_share / 2, 0.38, "answer\nnever seen", ha="center", fontsize=8.5)
axr.text(unseen_share + (1 - unseen_share) / 2, 0.38, "answer seen\nin training", ha="center", fontsize=8.5)
axr.set_xlim(0, 1); axr.set_ylim(-0.6, 0.8); axr.axis("off")
axr.set_title(f"All {len(errs):,} errors: where do they come from?")
axr.text(0, -0.55, f"{pct(in_train)} of the wrong outputs are themselves answers\nfrom the training data – it picks a familiar answer.",
         fontsize=8.5, color="#555555")
headline(fig, "Unseen answers fail 2-3x more often – yet they cause only a small share of the errors",
         "Memorization matters, but most failures happen on answers the model has seen many times.")
save(fig, "04_memorization")

## 5 · Most mistakes are near-misses

*What you see:* every wrong output, classified by how it differs from the correct answer (from `04`). Blue = small
slip (the answer has the right shape), grey = structurally different answer.

In [ ]:
def blocks(expr):
    """Linear MBA -> {bitwise term (by truth table) or 'const': coefficient}."""
    feats = Counter()

    def walk(n, sign):
        if isinstance(n, ast.BinOp) and isinstance(n.op, (ast.Add, ast.Sub)):
            walk(n.left, sign); walk(n.right, sign if isinstance(n.op, ast.Add) else -sign)
        elif nr._const(n):
            feats["const"] += sign * nr.evaluate(n, {})
        elif isinstance(n, ast.UnaryOp) and isinstance(n.op, (ast.USub, ast.UAdd)):
            walk(n.operand, -sign if isinstance(n.op, ast.USub) else sign)
        elif nr._bitwise(n):
            names = sorted(set(re.findall(r"[a-z]", ast.unparse(n))))
            tt = "".join(str(nr.evaluate(n, dict(zip(names, b))) & 1) for b in itertools.product((0, 1), repeat=len(names)))
            feats["".join(names) + ":" + tt] += sign
        elif isinstance(n, ast.BinOp) and isinstance(n.op, ast.Mult) and nr._const(n.left):
            walk(n.right, sign * nr.evaluate(n.left, {}))
        elif isinstance(n, ast.BinOp) and isinstance(n.op, ast.Mult) and nr._const(n.right):
            walk(n.left, sign * nr.evaluate(n.right, {}))
        else:
            raise ValueError(expr)

    walk(nr.parse(expr), 1)
    return feats


def off_by_one(pred, trg):
    """True if the prediction differs from the answer in exactly one coefficient, by exactly 1."""
    try:
        d = blocks(pred)
        d.subtract(blocks(trg))
    except ValueError:
        return False
    d = {k: v for k, v in d.items() if v}
    return len(d) == 1 and abs(next(iter(d.values()))) == 1

mist = tax.merge(test[["row", "trg", "pred"]], on="row")
coef = mist[mist.category == "wrong coefficient"]
obo = np.mean([off_by_one(p, t) for p, t in zip(coef.pred, coef.trg)])

SMALL = ["wrong coefficient", "wrong bitwise function", "swapped variables", "off by a constant", "sign flip"]
LABEL = {"wrong coefficient": "wrong coefficient  (2*(x&y) → 3*(x&y))", "wrong bitwise function": "wrong bitwise function  (x&y → x|y)",
         "swapped variables": "swapped variables  (x&~y → ~x&y)", "off by a constant": "off by a constant  (x+1 → x)",
         "sign flip": "sign flip  (x^y → -(x^y))", "missing term": "a term missing", "extra term": "an extra term",
         "several changes": "several changes", "non-linear output": "non-linear output"}
counts = mist.category.value_counts()
cats = [c for c in SMALL if c in counts] + [c for c in ["missing term", "extra term", "several changes"] if c in counts]
vals = counts.reindex(cats)

fig, ax = plt.subplots(figsize=(10, 4))
y = np.arange(len(cats))
ax.barh(y, vals, color=[C["accent"] if c in SMALL else C["neutral"] for c in cats], height=0.65)
for yi, v in zip(y, vals):
    ax.text(v + 4, yi, f"{v}  ({pct(v / len(mist))})", va="center", fontsize=9)
ax.set_yticks(y, [LABEL[c] for c in cats])
ax.invert_yaxis()
ax.set_xlim(0, vals.max() * 1.22)
ax.set_xlabel("wrong outputs")
ax.grid(axis="y", visible=False)
small_share = mist.category.isin(SMALL).mean()
headline(fig, f"{pct(small_share)} of mistakes are small slips – the answer has the right shape",
         f"Of the wrong coefficients, {pct(obo)} are off by exactly 1 on a single term.")
save(fig, "05_mistakes")

## 6 · Two-variable AND/OR answers get collapsed to one variable

*What you see:* for the most common short answers that combine two variables, what the model answered.
The top bar is the comparison: all answers that are a single variable (`x`, `~y`, `-2*x`, …).

In [ ]:
def outcome(r):
    if r.correct:
        return "correct"
    pv = set(re.findall(r"[a-z]", r.pred))
    if len(pv) == 1:
        return "collapsed to one variable"
    if r.pred_n_terms == 1 and r.pred_max_const == 0:
        return "a different AND/OR/XOR of both"
    return "other mistake"

short = test[(test.trg_n_terms == 1) & (test.trg_len <= 5)].copy()
short["outcome"] = short.apply(outcome, axis=1)
single = short[short.trg_n_vars == 1]
two = short[(short.trg_n_vars == 2) & (short.trg_max_const == 0)]
top = two.trg.value_counts()
top = top[top >= 40].index[:6]

rows = [("single-variable answers", single)] + [(t, two[two.trg == t]) for t in top]
OUTS = [("correct", C["correct"]), ("collapsed to one variable", C["wrong"]),
        ("a different AND/OR/XOR of both", C["warm"]), ("other mistake", C["neutral"])]
fig, ax = plt.subplots(figsize=(10, 0.55 * len(rows) + 1.6))
for yi, (name, d) in enumerate(rows):
    left = 0
    for o, color in OUTS:
        share = (d.outcome == o).mean()
        ax.barh(yi, share, left=left, color=color, height=0.62, label=o if yi == 0 else None)
        if share >= 0.06:
            ax.text(left + share / 2, yi, pct(share), ha="center", va="center", fontsize=8.5,
                    color="white" if o != "other mistake" else "#333333", fontweight="bold")
        left += share
    ax.text(1.01, yi, f"n = {len(d):,}", va="center", fontsize=8.5, color="#555555")
ax.set_yticks(range(len(rows)), [r[0] for r in rows], fontfamily="monospace")
ax.get_yticklabels()[0].set_fontfamily("sans-serif")
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(lambda v, _: pct(v))
ax.grid(False)
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.08), fontsize=8.5)
headline(fig, "Answers like x&y fail up to a third of the time – often collapsed to a single variable",
         "Single-variable answers are almost always right; combining two variables is where it breaks (x&y → x, x|~y → ~y).")
save(fig, "06_bitwise")

## 7 · Confidence flags about half the errors – the rest are made confidently

*What you see:* if we flag every output whose least certain character has a probability below a threshold (x-axis):
how many of the errors we catch, and how many outputs we flag in total.

In [ ]:
th = np.linspace(0.3, 1.0, 71)
caught = [test.error[test.min_tok_prob < t].sum() / test.error.sum() for t in th]
flagged = [(test.min_tok_prob < t).mean() for t in th]
confident_wrong = (test.min_tok_prob[test.error] >= 0.9).mean()

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(th, caught, color=C["wrong"], lw=2.2, label="errors caught")
ax.plot(th, flagged, color=C["accent"], lw=2.2, label="outputs flagged")
k = int(np.argmin(np.abs(th - 0.7)))
ax.axvline(0.7, color="#999999", ls=":", lw=1)
ax.annotate(f"threshold 0.7: flags {pct(flagged[k])} of outputs,\ncatches {pct(caught[k])} of errors",
            (0.7, caught[k]), xytext=(0.36, 0.78), fontsize=9, arrowprops=dict(arrowstyle="->", color="#777777"))
ax.set_xlabel("flag an output if its least certain character is below …")
ax.yaxis.set_major_formatter(lambda v, _: pct(v))
ax.set_ylim(0, 1.02)
ax.legend(loc="center left")
headline(fig, "The model is less sure when it is wrong – but not reliably",
         f"{pct(confident_wrong)} of errors are made with every character above 90% probability; "
         "AND/OR confusions are the most confident mistakes.")
save(fig, "07_confidence")

## 8 · Rewrites that keep the math the same break most correct answers

*What you see:* we took inputs the model solved correctly, rewrote them without changing their meaning (`05`), and
checked whether the model still gets them right. Each change was verified mathematically.

In [ ]:
EXAMPLES = {"control (no change)": "unchanged input", "rename x -> c": "x → c   (c with y is common in training)",
            "swap operands": "x&~y → ~y&x", "swap x and y": "every x ↔ y",
            "add cancelling terms": "+(x&y)-(x&y) inserted", "reorder terms": "a - b + c → c + a - b",
            "add zero identity": "+(x^y)-(x|y)+(x&y) inserted (= 0)", "rename x,y -> a,b": "x, y → a, b   (pair never seen)",
            "answer as input": "the correct answer itself as input"}
p = probes[probes.orig == "correct"]
t = rates(p, "probe", col="ok").sort_values("rate", ascending=False)
t = pd.concat([t.loc[["control (no change)"]], t.drop("control (no change)")])

fig, ax = plt.subplots(figsize=(10, 4.4))
y = np.arange(len(t))
colors = [C["neutral"]] + [GOOD_BAD(r) for r in t.rate.iloc[1:]]
ax.barh(y, t.rate, color=colors, height=0.66)
ax.errorbar(t.rate, y, xerr=[t.rate - t.lo, t.hi - t.rate], fmt="none", ecolor="#333333", capsize=2, lw=0.8)
for yi, r, hi in zip(y, t.rate, t.hi):
    ax.text(hi + 0.015, yi, pct(r), va="center", fontweight="bold", fontsize=9)
ax.set_yticks(y, [f"{name}\n{EXAMPLES.get(name, '')}" for name in t.index], fontsize=8.5)
ax.invert_yaxis()
ax.set_xlim(0, 1.1)
ax.xaxis.set_major_formatter(lambda v, _: pct(v))
ax.set_xlabel(f"previously correct answers that are still correct (n = {int(t.n.max()):,} per change, fewer where a change doesn't apply)")
ax.grid(axis="y", visible=False)
headline(fig, "Same math, different form: most correct answers break",
         "Only changes that keep the input looking like training data are harmless.")
save(fig, "08_rewrites")

## 9 · The fragility is everywhere, not in a corner of the data

*What you see:* the same "still correct" share, split by kind of input. Same colour scale as before.

In [ ]:
pp = p.merge(test[["row", "input_type"]], on="row")
pp["group"] = pp.input_type
pp.loc[(pp.trg_n_terms == 1) & (pp.trg_len <= 5) & (pp.trg_n_vars == 2), "group"] = "two-variable AND/OR answer"
groups = ["variables x, y", "two-variable AND/OR answer", "other variable pair", "three variables"]
changes = [c for c in t.index if c != "control (no change)"]
grid = pp[pp.probe != "control (no change)"].pivot_table(index="group", columns="probe", values="ok", aggfunc="mean")
ns = pp[pp.probe == "reorder terms"].group.value_counts()
counts = pp[pp.probe != "control (no change)"].pivot_table(index="group", columns="probe", values="ok", aggfunc="size")
grid = grid.where(counts >= 20)  # hide cells based on fewer than 20 inputs
grid = grid.reindex(index=[g for g in groups if g in grid.index], columns=changes)

fig, ax = plt.subplots(figsize=(11, 0.6 * len(grid) + 2.2))
im = ax.imshow(grid.to_numpy(float), cmap=GOOD_BAD, vmin=0, vmax=1, aspect="auto")
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        v = grid.iat[i, j]
        if not np.isnan(v):
            ax.text(j, i, pct(v), ha="center", va="center", fontsize=9, fontweight="bold",
                    color="white" if v < 0.25 or v > 0.8 else "#222222")
ax.set_xticks(range(len(changes)), changes, rotation=30, ha="right")
ax.set_yticks(range(len(grid)), [f"{g}  (n={ns.get(g, 0):,})" for g in grid.index])
ax.grid(False)
cb = fig.colorbar(im, ax=ax, fraction=0.025)
cb.ax.yaxis.set_major_formatter(lambda v, _: pct(v))
headline(fig, "Every kind of input shows the same pattern",
         "Correct does not mean robust: the model is right only as long as the input is written the usual way. "
         "Empty cells: change not applicable or fewer than 20 inputs.")
save(fig, "09_fragile_everywhere")

## 10 · With unfamiliar variable names, the model invents variables

*What you see:* after renaming the variables, how often the output contains a variable that is **not in the input**
(e.g. answering with `x` when the input only uses `a` and `b`). Below: real examples.

In [ ]:
def invents(r):
    return bool(set(re.findall(r"[a-z]", r.new_pred)) - set(re.findall(r"[a-z]", r.new_src)))

ren = probes[probes.probe.isin(["rename x -> c", "rename x,y -> a,b"])].copy()
ren["invented"] = ren.apply(invents, axis=1)
t = rates(ren, "probe", col="invented").reindex(["rename x -> c", "rename x,y -> a,b"])

fig, (ax, axt) = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw={"width_ratios": [1, 1.5]})
labels = ["x → c\n(familiar pair)", "x, y → a, b\n(pair never seen)"]
ax.bar(labels, t.rate, color=[C["accent"], C["wrong"]], width=0.55)
for i, (r, n) in enumerate(zip(t.rate, t.n)):
    ax.text(i, r + 0.01, f"{pct(r)}\nn={n:,}", ha="center", fontweight="bold", fontsize=9)
ax.set_ylim(0, max(0.2, t.rate.max() * 1.35))
ax.yaxis.set_major_formatter(lambda v, _: pct(v))
ax.set_ylabel("outputs with an invented variable")

ex = ren[ren.invented & (ren.probe == "rename x,y -> a,b") & (ren.new_src.str.len() <= 40)].head(4)
axt.axis("off")
axt.text(0, 0.95, "input (after renaming)", fontweight="bold", fontsize=9, transform=axt.transAxes)
axt.text(0.62, 0.95, "model output", fontweight="bold", fontsize=9, transform=axt.transAxes)
for k, (_, r) in enumerate(ex.iterrows()):
    yy = 0.78 - k * 0.2
    axt.text(0, yy, r.new_src, family="monospace", fontsize=8, transform=axt.transAxes)
    axt.text(0.62, yy, r.new_pred, family="monospace", fontsize=8.5, color=C["wrong"], transform=axt.transAxes)
headline(fig, "Variable names are not placeholders: with unseen names the model falls back to x and y",
         "It sometimes writes even meaningless terms such as ~b&b (always 0).")
save(fig, "10_invented_variables")

## 11 · The training data has a fixed operand order – and the model learned it

*What you see:* inside every bitwise operation in the training inputs (`a&b`, `a|b`, `a^b`), how often `x` and `y`
stand on the left or on the right.

In [ ]:
pos = Counter()
for s in train.src:
    for a, b in re.findall(r"~?\(?~?([a-z])[&|^]\(?~?([a-z])", s):
        if a != b:
            pos["x", "left"] += a == "x"; pos["x", "right"] += b == "x"
            pos["y", "left"] += a == "y"; pos["y", "right"] += b == "y"

fig, ax = plt.subplots(figsize=(8, 3.4))
xs = np.arange(2)
w = 0.36
for i, (side, color) in enumerate([("left", C["accent"]), ("right", C["warm"])]):
    vals = [pos["x", side], pos["y", side]]
    ax.bar(xs + (i - 0.5) * w, vals, width=w, color=color, label=f"{side} operand")
    for xi, v in zip(xs, vals):
        ax.text(xi + (i - 0.5) * w, v + 2000, f"{v:,}", ha="center", fontsize=9, fontweight="bold")
ax.set_xticks(xs, ["variable x", "variable y"])
ax.set_ylim(0, max(pos.values()) * 1.15)
ax.yaxis.set_major_formatter(lambda v, _: f"{v / 1000:.0f}k")
ax.set_ylabel("occurrences in training inputs")
ax.legend(loc="upper center")
swap = rates(p, "probe", col="ok")
headline(fig, "x is never written on the right – so the model has never seen y&x",
         f"Swapping operands keeps only {pct(swap.rate['swap operands'])} of correct answers, "
         f"swapping x and y only {pct(swap.rate['swap x and y'])}.")
save(fig, "11_convention")

## 12 · On a map of all inputs, only the three-variable inputs stand out

*What you see:* every test input, placed so that inputs that look alike are close (UMAP of their character patterns,
variable names normalized). Red = wrong output. Takes about a minute.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import umap

def normalize_names(expr):
    names = set(re.findall(r"[a-z]", expr))
    free = [v for v in "xy" if v not in names]
    return expr.translate(str.maketrans(dict(zip(sorted(names - {"x", "y", "z"}), free))))

X = TfidfVectorizer(analyzer="char", ngram_range=(2, 4), min_df=5, sublinear_tf=True).fit_transform(test.src.map(normalize_names))
Z = TruncatedSVD(n_components=40, random_state=0).fit_transform(X)
test[["mx", "my"]] = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=0, n_jobs=1).fit_transform(Z)

fig, ax = plt.subplots(figsize=(10, 6.5))
ok, bad = test[~test.error], test[test.error]
ax.scatter(ok.mx, ok.my, s=3, color=C["neutral"], label=f"correct ({len(ok):,})")
ax.scatter(bad.mx, bad.my, s=6, color=C["wrong"], label=f"wrong ({len(bad):,})")
three = test[test.input_type == "three variables"]
ax.annotate(f"three variables\n{pct(three.error.mean())} wrong", (three.mx.median(), three.my.median()),
            xytext=(25, 30), textcoords="offset points", fontsize=9.5, fontweight="bold", color=C["wrong"],
            arrowprops=dict(arrowstyle="->", color=C["wrong"]))
rest = test[test.input_type != "three variables"]
ax.text(0.99, 0.02, f"everything else: {pct(rest.error.mean())} wrong,\nspread across the whole map",
        transform=ax.transAxes, ha="right", fontsize=9.5, color="#444444")
ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
ax.legend(loc="upper left", markerscale=3)
headline(fig, "Errors are not concentrated in a region of input space",
         "What decides success is the exact written form of an input (term order, variable position), not its type.")
save(fig, "12_map")

## Summary

| | Conclusion | Evidence |
|---|---|---|
| 1 | The model only handles inputs in the **form** it was trained on | rewrites break 40-90% of correct answers (fig. 8, 9); operand-order convention (fig. 11) |
| 2 | Rare kinds of input fail | three variables: ~47% wrong vs. ~9% (fig. 2, 12) |
| 3 | Combining two variables with AND/OR is a specific weakness | x&y, x\|~y collapsed to one variable (fig. 6) |
| 4 | Mistakes are near-misses, often confident | ~3/4 small slips, mostly off by one (fig. 5, 7) |
| 5 | Memorization matters, but is not the main cause | unseen answers fail 2-3x more, yet cause ~10% of errors (fig. 4) |
| 6 | Variable names act as fixed tokens | invented variables with unseen names (fig. 10) |

**Caveats:** the test set was used for early stopping during training (strictly a validation set); three-variable
inputs are only 102 test examples; rewritten inputs differ in form from all training data, so fig. 8 shows a lack of
robustness to equivalent rewrites, not that the model "does no math".